In [7]:
from pathlib import Path

print("Current directory:", Path.cwd())

project_folder = Path.home() / "Desktop" / "ACA_Capstone"

print("Project folder exists:", project_folder.exists())

csv_files = sorted(project_folder.rglob("*.csv"))

print("Total CSV files found:", len(csv_files))

for file in csv_files:
    print(file.resolve())

Current directory: C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\notebooks
Project folder exists: True
Total CSV files found: 11
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\categories.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\customers.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\employee_territories.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\employees.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\order_details.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\orders.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\products.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\regions.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\shippers.csv
C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-caps

In [8]:
%pip install pandas sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 2.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------- ----------------------------- 0.8/2.9 MB 6.5 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.9 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 4.8 MB/s  0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ----------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import ipywidgets as widgets
from IPython.display import display

password_box = widgets.Password(
    description="Password:",
    placeholder="Enter PostgreSQL password"
)

display(password_box)

Password(description='Password:', placeholder='Enter PostgreSQL password')

In [2]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

db_url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password_box.value,
    host="localhost",
    port=5432,
    database="northwind_capstone"
)

engine = create_engine(db_url)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database()"))
        print("Connected to:", result.scalar())
finally:
    password_box.value = ""

Connected to: northwind_capstone


In [4]:
from pathlib import Path
import pandas as pd

project_folder = Path.home() / "Desktop" / "ACA_Capstone"

csv_files = sorted(project_folder.rglob("*.csv"))

print("CSV files found:", len(csv_files))

orders_path = next(
    file for file in csv_files
    if file.name == "orders.csv"
)

print("Orders path:", orders_path)

orders = pd.read_csv(
    orders_path,
    dtype=str,
    keep_default_na=False
)

print("Orders rows:", len(orders))
print("Orders columns:", orders.columns.tolist())

display(orders.head())

CSV files found: 11
Orders path: C:\Users\DEVOPS\Desktop\ACA_Capstone\northwind-capstone-pipeline\data\row\orders.csv
Orders rows: 830
Orders columns: ['orderID', 'customerID', 'employeeID', 'orderDate', 'requiredDate', 'shippedDate', 'shipVia', 'freight', 'shipName', 'shipAddress', 'shipCity', 'shipRegion', 'shipPostalCode', 'shipCountry']


,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,VINET,5,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NULL,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NULL,44087,Germany
2,10250,HANAR,4,1996-07-08 00:00:00.000,1996-08-05 00:00:00.000,1996-07-12 00:00:00.000,2,65.83,Hanari Carnes,Rua do Paço 67,Rio de Janeiro,RJ,05454-876,Brazil
3,10251,VICTE,3,1996-07-08 00:00:00.000,1996-08-05 00:00:00.000,1996-07-15 00:00:00.000,1,41.34,Victuailles en stock,2 rue du Commerce,Lyon,NULL,69004,France
4,10252,SUPRD,4,1996-07-09 00:00:00.000,1996-08-06 00:00:00.000,1996-07-11 00:00:00.000,2,51.30,Suprêmes délices,Boulevard Tirou 255,Charleroi,NULL,B-6000,Belgium


In [5]:
from sqlalchemy import inspect, text
import pandas as pd

raw_folder = orders_path.parent

tables = [
    "categories",
    "customers",
    "employee_territories",
    "employees",
    "order_details",
    "orders",
    "products",
    "regions",
    "shippers",
    "suppliers",
    "territories"
]

# Check source files and database tables
db_tables = set(inspect(engine).get_table_names(schema="stg"))
assert set(tables).issubset(db_tables), "Missing staging tables"

data = {}

for table in tables:
    file_path = raw_folder / f"{table}.csv"

    df = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False
    )

    db_columns = [
        col["name"]
        for col in inspect(engine).get_columns(table, schema="stg")
    ]

    assert list(df.columns) == db_columns, f"Column mismatch: {table}"
    data[table] = df

# Load all tables in one transaction
with engine.begin() as conn:
    for table in tables:
        conn.execute(text(f'TRUNCATE TABLE stg."{table}"'))

    for table, df in data.items():
        df.to_sql(
            name=table,
            con=conn,
            schema="stg",
            if_exists="append",
            index=False,
            chunksize=500,
            method="multi"
        )

        count = conn.execute(
            text(f'SELECT COUNT(*) FROM stg."{table}"')
        ).scalar_one()

        assert count == len(df), f"Row count mismatch: {table}"
        print(f"{table}: {count} rows loaded")

print("Total rows:", sum(len(df) for df in data.values()))
print("Staging load completed successfully!")

categories: 8 rows loaded
customers: 91 rows loaded
employee_territories: 49 rows loaded
employees: 9 rows loaded
order_details: 2155 rows loaded
orders: 830 rows loaded
products: 77 rows loaded
regions: 4 rows loaded
shippers: 3 rows loaded
suppliers: 29 rows loaded
territories: 53 rows loaded
Total rows: 3308
Staging load completed successfully!


In [6]:
import sys

project_root = orders_path.parents[2]

sys.path.insert(0, str(project_root))

from src.stage_raw_data import load_staging

counts = load_staging(engine, orders_path.parent)

categories: 8 rows loaded
customers: 91 rows loaded
employee_territories: 49 rows loaded
employees: 9 rows loaded
order_details: 2155 rows loaded
orders: 830 rows loaded
products: 77 rows loaded
regions: 4 rows loaded
shippers: 3 rows loaded
suppliers: 29 rows loaded
territories: 53 rows loaded
Total rows: 3308
Staging load completed successfully!


In [7]:
first_run = load_staging(engine, orders_path.parent)
second_run = load_staging(engine, orders_path.parent)

assert first_run == second_run, "Row counts changed!"

print("Idempotency row-count check: PASSED")
print("Total rows:", sum(second_run.values()))

categories: 8 rows loaded
customers: 91 rows loaded
employee_territories: 49 rows loaded
employees: 9 rows loaded
order_details: 2155 rows loaded
orders: 830 rows loaded
products: 77 rows loaded
regions: 4 rows loaded
shippers: 3 rows loaded
suppliers: 29 rows loaded
territories: 53 rows loaded
Total rows: 3308
Staging load completed successfully!
categories: 8 rows loaded
customers: 91 rows loaded
employee_territories: 49 rows loaded
employees: 9 rows loaded
order_details: 2155 rows loaded
orders: 830 rows loaded
products: 77 rows loaded
regions: 4 rows loaded
shippers: 3 rows loaded
suppliers: 29 rows loaded
territories: 53 rows loaded
Total rows: 3308
Staging load completed successfully!
Idempotency row-count check: PASSED
Total rows: 3308
